<a href="https://colab.research.google.com/github/ronjaladhikari213/AI-related-projects/blob/main/Cancer_Detection_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [73]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [74]:
df=pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [75]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [76]:
X_train,X_test,y_train,y_test=train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)
X_test.shape

(114, 30)

In [77]:
scalar=StandardScaler()
X_train=scalar.fit_transform(X_train)
X_test=scalar.transform(X_test)

encoder=LabelEncoder()
y_train=encoder.fit_transform(y_train)
y_test=encoder.transform(y_test)

In [78]:
X_train_tensor=torch.from_numpy(X_train)
X_test_tensor=torch.from_numpy(X_test)
y_train_tensor=torch.from_numpy(y_train)
y_test_tensor=torch.from_numpy(y_test)

In [79]:
class MyNN():
  def __init__(self,X):
    self.weights=torch.rand(X_test.shape[1],1,dtype=torch.float64,requires_grad=True)
    self.bias=torch.zeros(1,dtype=torch.float64,requires_grad=True)

  def forward(self,X):
    z=torch.matmul(X,self.weights)+self.bias
    y_pred=torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss


In [80]:
lr=0.01
epochs=200

In [84]:
model=MyNN(X_train_tensor)

#Forward propagation
for epoch in range(epochs):
  y_pred=model.forward(X_train_tensor)

  #loss
  loss_function=model.loss_function(y_pred,y_train_tensor)

  #backward propagation
  loss_function.backward()

  #Updating the parameters
  with torch.no_grad():
    model.weights-=lr*model.weights.grad
    model.bias-=lr*model.bias.grad

  #Zeroing the gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

In [82]:
#model evaluation
with torch.no_grad():
  y_pred=model.forward(X_test_tensor)
  print(y_pred)

tensor([[0.3509],
        [0.3744],
        [0.3426],
        [0.3516],
        [0.3928],
        [0.3618],
        [0.3644],
        [0.3565],
        [0.3934],
        [0.3834],
        [0.3715],
        [0.4044],
        [0.3657],
        [0.3661],
        [0.3758],
        [0.3680],
        [0.3652],
        [0.3894],
        [0.3815],
        [0.3698],
        [0.4035],
        [0.3682],
        [0.4020],
        [0.3743],
        [0.4039],
        [0.3807],
        [0.3580],
        [0.3644],
        [0.3625],
        [0.3676],
        [0.3646],
        [0.2572],
        [0.3578],
        [0.4821],
        [0.3809],
        [0.3142],
        [0.3791],
        [0.3813],
        [0.3522],
        [0.3786],
        [0.3628],
        [0.3809],
        [0.4053],
        [0.3610],
        [0.3508],
        [0.4006],
        [0.3651],
        [0.4147],
        [0.3539],
        [0.3542],
        [0.3804],
        [0.3831],
        [0.3780],
        [0.3686],
        [0.3770],
        [0